# Orquestrando Agentes — ReAct com ChatOpenAI

Este notebook reorganiza as primeiras aulas do curso e substitui o **Gemini** por **`ChatOpenAI`**, mantendo a lógica didática do agente ReAct.

## O que foi preservado

- Estado do agente com `TypedDict`
- Ferramentas de estoque e preço
- Ferramenta para encontrar o produto mais caro
- Ferramenta para calcular o valor total de uma lista
- Ciclo **Ação → Observação → Resposta**
- Parser das ações com expressão regular
- Histórico das mensagens
- Loop interativo no terminal/notebook

## O que mudou

- `google.generativeai` → `langchain_openai.ChatOpenAI`
- `GEMINI_API_KEY` → `OPENAI_API_KEY`
- `model.start_chat()` / `chat.send_message()` → `llm.invoke(mensagens)`
- Histórico passa a usar `SystemMessage`, `HumanMessage` e `AIMessage`

> **Observação:** `StateGraph` e `END` são importados porque aparecem no curso, mas nesta etapa o agente ReAct ainda é orquestrado manualmente. O grafo do LangGraph será usado quando as aulas chegarem à construção efetiva do `StateGraph`.


## 1. Instalação das dependências

In [ ]:
%pip install -U langchain-openai langchain-community langgraph python-dotenv

## 2. Imports

O `ChatOpenAI` é a implementação do LangChain para os modelos da OpenAI.


In [ ]:
import os
import re
from typing import TypedDict

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

from langgraph.graph import StateGraph, END


## 3. Configuração das variáveis de ambiente

Crie um arquivo `.env` na mesma pasta do notebook:

```env
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=...
```

A `TAVILY_API_KEY` ainda não é usada nestas primeiras aulas, mas pode ser mantida porque será útil quando o curso entrar em busca na web.

Você também pode escolher outro modelo sem editar o código:

```env
OPENAI_MODEL=gpt-5.6-luna
```


In [ ]:
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY não encontrada. "
        "Crie um arquivo .env e adicione OPENAI_API_KEY=sua_chave."
    )

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")

print(f"Modelo configurado: {MODEL}")


## 4. Criando o modelo com `ChatOpenAI`

In [ ]:
llm = ChatOpenAI(
    model=MODEL
)

print(llm)


## 5. Teste da API — Hello World

No material original, o teste era feito com `GenerativeModel(...).generate_content(...)`.

Com `ChatOpenAI`, usamos `invoke()`.


In [ ]:
response = llm.invoke("Hello world")

print(response.content)


## 6. Função auxiliar para extrair texto de uma mensagem

Na maioria das respostas de texto, `AIMessage.content` é uma `str`. A função abaixo também lida com respostas representadas como blocos de conteúdo.


In [ ]:
def extrair_texto(mensagem: AIMessage) -> str:
    conteudo = mensagem.content

    if isinstance(conteudo, str):
        return conteudo.strip()

    partes = []

    if isinstance(conteudo, list):
        for bloco in conteudo:
            if isinstance(bloco, str):
                partes.append(bloco)
            elif isinstance(bloco, dict):
                texto = bloco.get("text")
                if texto:
                    partes.append(texto)

    return "\n".join(partes).strip()


## 7. Classe `Agent`

Esta é a adaptação da classe inicial das aulas para `ChatOpenAI`.

Ela mantém um histórico de mensagens e envia esse histórico ao modelo.


In [ ]:
class Agent:
    def __init__(self, system: str = ""):
        self.system = system
        self.messages = []

    def __call__(self, message: str) -> str:
        self.messages.append(HumanMessage(content=message))

        result = self.execute()

        self.messages.append(AIMessage(content=result))

        return result

    def execute(self) -> str:
        mensagens = []

        if self.system:
            mensagens.append(SystemMessage(content=self.system))

        mensagens.extend(self.messages)

        response = llm.invoke(mensagens)

        return extrair_texto(response)


In [ ]:
# Teste da classe Agent
agent = Agent(system="Você é um assistente útil e objetivo.")

print(agent("Qual é a capital da França?"))


## 8. Estado do agente

A estrutura abaixo é a mesma ideia ensinada no curso. Ela será especialmente útil quando o fluxo passar a ser representado por um `StateGraph`.


In [ ]:
class EstadoAgente(TypedDict):
    pergunta: str
    historico: list[str]
    acao_pendente: str
    resposta_final: str


## 9. Dados do inventário

Para evitar repetir os mesmos dados em várias funções, estoque e preços ficam centralizados.


In [ ]:
ESTOQUE = {
    "monitor": 75,
    "teclado": 120,
    "mouse gamer": 80,
    "webcam": 40,
    "headset": 60,
    "impressora": 15,
}

PRECOS = {
    "monitor": 999.90,
    "teclado": 150.00,
    "mouse gamer": 99.50,
    "webcam": 120.00,
    "headset": 180.00,
    "impressora": 750.00,
}


## 10. Ferramenta: consultar estoque

In [ ]:
def consultar_estoque(item: str) -> str:
    """
    Simula a consulta de estoque de um item no inventário.
    """
    item = item.lower().strip()

    if item in ESTOQUE:
        return f"Temos {ESTOQUE[item]} {item}s em estoque."

    return f"Item '{item}' não encontrado no inventário."


## 11. Ferramenta: consultar preço

In [ ]:
def consultar_preco_produto(produto: str) -> str:
    """
    Simula a consulta do preço unitário de um produto.
    """
    produto = produto.lower().strip()

    if produto in PRECOS:
        return f"O preço de um(a) {produto} é R$ {PRECOS[produto]:.2f}."

    return f"Produto '{produto}' não encontrado na lista de preços."


## 12. Testes das ferramentas iniciais

In [ ]:
print(consultar_estoque("teclado"))
print(consultar_preco_produto("impressora"))
print(consultar_estoque("monitor"))


## 13. Ferramenta: encontrar o produto mais caro

In [ ]:
def ferramenta_encontrar_produto_mais_caro() -> str:
    """
    Retorna o nome e o preço do produto mais caro no inventário.
    Esta função não precisa de argumentos.
    """
    if not PRECOS:
        return "Nenhum produto encontrado na lista de preços para comparação."

    nome_produto_mais_caro = max(PRECOS, key=PRECOS.get)
    valor_produto_mais_caro = PRECOS[nome_produto_mais_caro]

    return (
        f"O produto mais caro é o(a) {nome_produto_mais_caro} "
        f"com preço de R$ {valor_produto_mais_caro:.2f}."
    )


In [ ]:
print(ferramenta_encontrar_produto_mais_caro())


## 14. Ferramenta: calcular o valor total de uma lista

In [ ]:
def ferramenta_calcular_valor_total_lista(lista_itens: str) -> str:
    """
    Calcula o valor total de uma lista de itens de compra.

    Recebe uma string com itens separados por vírgula.
    Exemplo:
        "teclado, mouse gamer, monitor"
    """
    itens_processados = [
        item.strip().lower()
        for item in lista_itens.split(",")
        if item.strip()
    ]

    valor_total = 0.0
    itens_nao_encontrados = []

    for item in itens_processados:
        if item in PRECOS:
            valor_total += PRECOS[item]
        else:
            itens_nao_encontrados.append(item)

    resposta = f"O valor total dos itens encontrados é R$ {valor_total:.2f}."

    if itens_nao_encontrados:
        resposta += (
            " Os seguintes itens não foram encontrados e não foram "
            "incluídos no cálculo: "
            + ", ".join(itens_nao_encontrados)
            + "."
        )

    return resposta


### Testando a ferramenta de soma

In [ ]:
lista_1 = "teclado, mouse gamer, monitor"
lista_2 = "headset, cadeira"
lista_3 = "mesa, copo"

print(f"Lista: {lista_1}")
print(ferramenta_calcular_valor_total_lista(lista_1))
print()

print(f"Lista: {lista_2}")
print(ferramenta_calcular_valor_total_lista(lista_2))
print()

print(f"Lista: {lista_3}")
print(ferramenta_calcular_valor_total_lista(lista_3))


## 15. Prompt do agente ReAct

O curso original usa **Pensamento → Ação → Pausa → Observação → Resposta**.

Aqui preservamos a ideia de **ReAct**, mas o protocolo textual necessário para o programa é reduzido a:

**Ação → PAUSA → Observação → Resposta**

O programa não precisa receber o raciocínio interno detalhado do modelo para conseguir selecionar e executar uma ferramenta.


In [ ]:
PROMPT_REACT = """
Você é um agente de inventário de uma loja de informática.

Seu objetivo é responder corretamente às perguntas do usuário usando,
quando necessário, as ferramentas disponibilizadas pelo programa.

Quando precisar usar uma ferramenta, responda EXATAMENTE no formato:

Ação: nome_da_ferramenta: argumento
PAUSA

Para uma ferramenta que não recebe argumento:

Ação: nome_da_ferramenta
PAUSA

Não invente o resultado da ferramenta.
Depois de solicitar uma Ação, aguarde a Observação enviada pelo programa.

Após receber uma Observação e ter informação suficiente para responder,
use EXATAMENTE:

Resposta: sua resposta final

Ações disponíveis:

- consultar_estoque:
  retorna a quantidade disponível de um item no inventário.
  Exemplo de chamada:
  Ação: consultar_estoque: teclado

- consultar_preco_produto:
  retorna o preço unitário de um produto.
  Exemplo de chamada:
  Ação: consultar_preco_produto: mouse gamer

- encontrar_produto_mais_caro:
  retorna o nome e o preço do produto mais caro no inventário.
  Não recebe argumento.
  Exemplo de chamada:
  Ação: encontrar_produto_mais_caro

- calcular_valor_total_lista:
  calcula o valor total de vários produtos.
  O argumento deve conter os nomes separados por vírgula.
  Exemplo de chamada:
  Ação: calcular_valor_total_lista: teclado, mouse gamer

Exemplo 1:

Pergunta: Quantos monitores temos em estoque?

Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.

Resposta: Há 75 monitores em estoque.


Exemplo 2:

Pergunta: Qual é o produto mais caro?

Ação: encontrar_produto_mais_caro
PAUSA

Observação: O produto mais caro é o(a) monitor com preço de R$ 999.90.

Resposta: O produto mais caro é o monitor, custando R$ 999.90.


Exemplo 3:

Pergunta: Quanto custa um teclado e um mouse gamer?

Ação: calcular_valor_total_lista: teclado, mouse gamer
PAUSA

Observação: O valor total dos itens encontrados é R$ 249.50.

Resposta: O valor total do teclado e do mouse gamer é R$ 249.50.
""".strip()

print(PROMPT_REACT)


## 16. Executor das ferramentas

Esta função representa a parte do **programa Python** que recebe o nome de uma ação escolhida pelo LLM e executa a função correspondente.


In [ ]:
def executar_ferramenta(action_name: str, action_arg: str = "") -> str:
    if action_name == "consultar_estoque":
        return consultar_estoque(action_arg)

    if action_name == "consultar_preco_produto":
        return consultar_preco_produto(action_arg)

    if action_name == "encontrar_produto_mais_caro":
        return ferramenta_encontrar_produto_mais_caro()

    if action_name == "calcular_valor_total_lista":
        return ferramenta_calcular_valor_total_lista(action_arg)

    return (
        f"Erro: Ação '{action_name}' desconhecida. "
        "Verifique o prompt ou a implementação da ferramenta."
    )


## 17. Função principal `run_react_agent`

Este é o coração do agente.

Fluxo:

1. A pergunta entra no histórico.
2. `ChatOpenAI` recebe o prompt do sistema e o histórico.
3. O GPT pode solicitar uma `Ação`.
4. Python interpreta essa ação.
5. Python executa a ferramenta.
6. A `Observação` volta ao modelo.
7. O ciclo continua até aparecer `Resposta:`.


In [ ]:
def run_react_agent(
    pergunta: str,
    max_iterations: int = 5,
    verbose: bool = True,
) -> str:

    historico = [
        SystemMessage(content=PROMPT_REACT),
        HumanMessage(content=f"Pergunta: {pergunta}"),
    ]

    for i in range(max_iterations):
        response = llm.invoke(historico)
        response_text = extrair_texto(response)

        if verbose:
            print(f"\n--- Iteração {i + 1} ---")
            print(f"Modelo respondeu:\n{response_text}\n")

        # Registra a resposta do modelo no histórico.
        historico.append(response)

        # Primeiro verificamos se o modelo forneceu a resposta final.
        response_match_final = re.search(
            r"Resposta:\s*(.*)",
            response_text,
            re.DOTALL | re.IGNORECASE,
        )

        if response_match_final:
            return response_match_final.group(1).strip()

        # Caso contrário, procuramos uma solicitação de ferramenta.
        match = re.search(
            r"Ação:\s*(\w+)(?::\s*([^\n]*))?",
            response_text,
            re.IGNORECASE,
        )

        if not match:
            return (
                "Erro: o agente não conseguiu identificar uma Ação "
                "ou uma Resposta final.\n\n"
                f"Última resposta do modelo:\n{response_text}"
            )

        action_name = match.group(1).strip()
        action_arg = (
            match.group(2).strip()
            if match.group(2) is not None
            else ""
        )

        observacao = executar_ferramenta(action_name, action_arg)

        if verbose:
            print(
                f"Executou ação: {action_name}"
                + (f"('{action_arg}')" if action_arg else "()")
            )
            print(f"Observação: {observacao}\n")

        # O resultado real da ferramenta volta ao modelo.
        historico.append(
            HumanMessage(content=f"Observação: {observacao}")
        )

    return (
        "Erro: limite máximo de iterações atingido "
        "sem uma resposta final."
    )


## 18. Testes completos do agente

In [ ]:
perguntas_teste = [
    "Quantos teclados temos em estoque?",
    "Qual o preço de um headset?",
    "Temos cadeiras em estoque?",
    "Qual é o produto mais caro?",
    "Qual o valor de um teclado, uma impressora e uma webcam?",
]

for numero, pergunta in enumerate(perguntas_teste, start=1):
    print("=" * 80)
    print(f"INTERAÇÃO {numero}: {pergunta}")

    resposta = run_react_agent(pergunta)

    print(f"\nRESPOSTA FINAL: {resposta}\n")


## 19. Versão que também retorna o histórico

Útil para estudar o ciclo completo entre:

- `SystemMessage`
- `HumanMessage`
- `AIMessage`
- Observações geradas pelas ferramentas


In [ ]:
def run_react_agent_with_history(
    pergunta: str,
    max_iterations: int = 5,
    verbose: bool = True,
) -> tuple[str, list]:

    historico = [
        SystemMessage(content=PROMPT_REACT),
        HumanMessage(content=f"Pergunta: {pergunta}"),
    ]

    for i in range(max_iterations):
        response = llm.invoke(historico)
        response_text = extrair_texto(response)

        if verbose:
            print(f"\n--- Iteração {i + 1} ---")
            print(f"Modelo respondeu:\n{response_text}\n")

        historico.append(response)

        response_match_final = re.search(
            r"Resposta:\s*(.*)",
            response_text,
            re.DOTALL | re.IGNORECASE,
        )

        if response_match_final:
            resposta_final = response_match_final.group(1).strip()
            return resposta_final, historico

        match = re.search(
            r"Ação:\s*(\w+)(?::\s*([^\n]*))?",
            response_text,
            re.IGNORECASE,
        )

        if not match:
            erro = (
                "Erro: o agente não conseguiu identificar uma Ação "
                "ou uma Resposta final."
            )
            return erro, historico

        action_name = match.group(1).strip()
        action_arg = (
            match.group(2).strip()
            if match.group(2) is not None
            else ""
        )

        observacao = executar_ferramenta(action_name, action_arg)

        if verbose:
            print(
                f"Executou ação: {action_name}"
                + (f"('{action_arg}')" if action_arg else "()")
            )
            print(f"Observação: {observacao}\n")

        historico.append(
            HumanMessage(content=f"Observação: {observacao}")
        )

    return (
        "Erro: limite máximo de iterações atingido sem resposta final.",
        historico,
    )


### Visualizando o histórico

In [ ]:
resposta_exemplo, historico_completo = run_react_agent_with_history(
    "Quantos teclados temos em estoque?"
)

print(f"\nRESPOSTA FINAL: {resposta_exemplo}\n")
print("--- HISTÓRICO COMPLETO ---")

for i, mensagem in enumerate(historico_completo, start=1):
    print(f"\nMensagem {i}")
    print(f"Tipo: {type(mensagem).__name__}")
    print(f"Conteúdo:\n{mensagem.content}")
    print("-" * 60)


## 20. Loop interativo

Execute esta célula quando quiser conversar continuamente com o agente.

Digite `sair` para encerrar.

> Cada pergunta chama `run_react_agent()` separadamente, portanto nesta versão o histórico de uma pergunta **não é preservado entre perguntas diferentes**. Persistência/memória será um passo posterior do curso.


In [ ]:
def iniciar_conversacao_com_agente():
    print("--- Agente de Inventário Interativo ---")
    print("Digite sua pergunta sobre o inventário.")
    print("Digite 'sair' para encerrar.")
    print("-" * 60)

    while True:
        pergunta_usuario = input("\nVocê: ")

        if pergunta_usuario.lower().strip() == "sair":
            print("Encerrando a conversa. Até logo!")
            break

        print("\nAgente: Processando...")

        try:
            resposta_agente = run_react_agent(
                pergunta_usuario,
                verbose=False,
            )

            print(f"Agente: {resposta_agente}")

        except Exception as erro:
            print(f"Agente: ocorreu um erro: {erro}")
            print("Tente novamente ou digite 'sair'.")


In [ ]:
# Descomente a linha abaixo para iniciar a conversa:
# iniciar_conversacao_com_agente()


# Resumo da conversão Gemini → ChatOpenAI

| Curso com Gemini | Notebook com ChatOpenAI |
|---|---|
| `google.generativeai` | `langchain_openai.ChatOpenAI` |
| `GEMINI_API_KEY` | `OPENAI_API_KEY` |
| `GenerativeModel(...)` | `ChatOpenAI(model=...)` |
| `start_chat()` | lista de mensagens LangChain |
| `chat.send_message(...)` | `llm.invoke(...)` |
| histórico Gemini | `SystemMessage`, `HumanMessage`, `AIMessage` |
| ferramentas Python | permanecem iguais |
| `TypedDict` | permanece igual |
| regex do ReAct | permanece |
| LangGraph | permanece e será usado nas próximas etapas |

## Arquitetura atual

```text
Usuário
   ↓
ChatOpenAI / GPT
   ↓
solicita Ação
   ↓
Python interpreta
   ↓
executa ferramenta
   ↓
Observação
   ↓
ChatOpenAI / GPT
   ↓
Resposta final
```

O objetivo desta versão é continuar acompanhando o raciocínio arquitetural do curso sem pular diretamente para `bind_tools()` ou para um agente pronto do LangGraph.
